# Deploying NVIDIA Nemotron 3.5 Lightning with SGLang

This notebook will walk you through how to run the NVIDIA Nemotron 3.5 Lightning NVFP4 checkpoint with SGLang on a single H100.

[SGLang](https://github.com/sgl-project/sglang) is a fast serving framework for large language models and vision language models.

Nemotron 3.5 Lightning is published as two checkpoints:

- **BF16** — [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16)
- **NVFP4** — [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4) - **this notebook deploys this checkpoint**

**Model size:** 30B total parameters, 3B active (MoE)

Prerequisites for this notebook:
- 1x NVIDIA H100 80GB with recent drivers
- Python 3.10+
- Docker 

## Overview

- **Serve** the Nemotron 3.5 Lightning NVFP4 checkpoint using SGLang
- **Query the model** through an OpenAI-compatible API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget

## Table of Contents

1. **Environment setup** - Dependencies and container image
2. **Verify GPU** - Confirm CUDA and GPU availability
3. **Start SGLang Server** - Base, MTP, DFlash, or DSpark configuration
4. **Generate responses** - Chat completions and streaming
   - **Client setup** - Point the OpenAI client at the server
   - **Single completion** - One chat request
   - **Streamed generation** - Receive tokens as they are produced
   - **Reasoning** - Thinking mode examples
   - **Tool calling** - Function calling via OpenAI tools schema
   - **Controlling Reasoning Budget** - Limit reasoning trace length
5. **Cleanup and shutdown**

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch the NVFP4 variant on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

**For NVFP4 (1x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Havtx6fAxwxMw9pLml5U6qAFrj)

## Environment setup

### Pull the SGLang Docker image

The model runs inside an SGLang container. Pull it once before starting the server:

```shell
docker pull lmsysorg/sglang:dev-nemotron3-5-lighting
```

### Install notebook client dependencies

These are for the notebook only: `openai` sends the requests, `transformers` provides the tokenizer used in the reasoning budget section, and `torch` backs the GPU check. The container already carries everything the model needs in order to load and run.

> **Note:** Match the `torch` build to your driver. A CUDA 13 wheel on a CUDA 12 driver reports `CUDA available: False` even when the GPU is healthy. In case that happens, check the driver's CUDA version with `nvidia-smi` and install from the matching index if needed, for example `--index-url https://download.pytorch.org/whl/cu128`. This affects only the GPU check below, not the model serving.

In [1]:
#If pip not found
!python3 -m ensurepip --default-pip

Looking in links: /tmp/tmpmf4ut1bx


In [ ]:
%pip install openai==2.38.0 transformers==5.9.0 torch

## Verify GPU

Confirm CUDA is available and your GPU is visible to PyTorch.

> **Expected output:** `CUDA available: True` with one H100 listed. If CUDA is `False`, check your driver installation.

In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

CUDA available: True
Num GPUs: 1
GPU[0]: NVIDIA H100 80GB HBM3


## Start SGLang Server

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the SGLang container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it \
  --gpus all \
  --cap-add SYS_NICE \
  --ipc=host \
  --network=host \
  --shm-size=16g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -e SAFETENSORS_FAST_GPU=1 \
  --entrypoint /bin/bash \
  lmsysorg/sglang:dev-nemotron3-5-lighting
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

All `sglang serve` commands below should be run from inside this container.

### Configuration reference

| | NVFP4 |
|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4` |
| **Hardware (this notebook)** | 1x H100 80GB |
| **Docker image** | `lmsysorg/sglang:dev-nemotron3-5-lighting` |
| **MoE runner backend** | `marlin` (auto-selected on H100) |
| **Mamba SSM cache** | FP16 |
| **Speculative decoding** | None, MTP, DFlash, or DSpark |
| **Reasoning parser** | `nemotron_3` |
| **Tool parser** | `qwen3_coder` |
| **Port** | 8000 |

### Start server

Run one of the following commands from inside the Docker container terminal. All four serve the same NVFP4 checkpoint and differ only in speculative decoding. Every later cell in this notebook works with any of them.

> **Note:** Parser names are backend-specific and not interchangeable. SGLang uses `--reasoning-parser nemotron_3` and `--tool-call-parser qwen3_coder`. vLLM uses `--reasoning-parser nemotron_v3` and TensorRT-LLM uses `--reasoning_parser nemotron-v3` — these are different identifiers for the same logical capability.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

#### MTP

MTP (Multi-Token Prediction) uses a small draft layer built into the model to guess several tokens ahead, which the model then verifies in one step. SGLang reaches that layer through its EAGLE path, so the draft model path is the target checkpoint itself.

> **Note:** Keep `--cuda-graph-max-bs-decode 16`. Without it, MTP's decode graphs reserve enough memory that short requests pass but a large-token request hits an out-of-memory error.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --speculative-algorithm EAGLE \
  --speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --speculative-num-steps 5 \
  --speculative-eagle-topk 1 \
  --speculative-num-draft-tokens 6 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

#### DFlash

DFlash does the same thing, but the guesses come from a separate draft model rather than a layer inside the model.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --speculative-algorithm DFLASH \
  --speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
  --speculative-dflash-block-size 4 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

#### DSpark

DSpark also uses a separate draft model, but it guesses a whole block of tokens at once rather than one at a time.

```shell
sglang serve --model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name nemotron-3.5-lightning \
  --mamba-ssm-dtype float16 \
  --mem-fraction-static 0.85 \
  --cuda-graph-max-bs-decode 16 \
  --speculative-algorithm DSPARK \
  --speculative-draft-model-path nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
  --speculative-dspark-block-size 3 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

### Wait for the server to be ready

After starting the container, poll `/v1/models` before sending any requests.

Run the following in a host terminal:

```shell
until curl -sf http://localhost:8000/v1/models > /dev/null 2>&1; do
  echo "Waiting for server..."; sleep 10
done
echo "Server is ready"
```

> **Expected output:** The loop prints `Waiting for server...` while the model loads, then exits and prints `Server is ready` once the server is accepting requests.

> **Note:** On the first run, the model weights will be downloaded from Hugging Face before loading begins, so the combined download and load time will be longer than on subsequent runs.

## Generate responses

> **Note:** Reasoning tokens count toward `max_tokens`. If `content` comes back empty or `None`, the reasoning trace consumed the entire budget before the model produced an answer — raise `max_tokens`.

### Client setup

In [1]:
from openai import OpenAI

# Set this to the model you launched the server with
SERVED_MODEL_NAME = "nemotron-3.5-lightning"
BASE_URL = "http://localhost:8000/v1"
API_KEY = "EMPTY"  # SGLang does not require an API key by default

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"OpenAI client configured to use server at: {BASE_URL}")
print(f"Using model: {SERVED_MODEL_NAME}")

OpenAI client configured to use server at: http://localhost:8000/v1
Using model: nemotron-3.5-lightning


### Single completion

In [4]:
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Give me 3 bullet points about SGLang."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)

Reasoning: Here I need to give 3 bullet points about SGLang. I need to produce some accurate, high-quality content. I must not give any prompts, just the facts. I need to interpret the user's request: "Give me 3 bullet points about SGLang." So I need to output three bullet points, each probably a few words or a sentence about what SGLang is. SGLang is a system for efficient execution of Transformer-like models. I need to be accurate. I'll provide 3 bullet points highlighting key features: efficient decoding, ease of use, and memory optimization. Probably should be mindful of formatting.

I need to give a concise and direct answer. I'll output three bullet points. I'll keep it simple: SGLang is an open-source language for difficult context or something? Actually, SGLang (Stable Language? No, SGLang is a model serving system). Let me recall: SGLang (Simple GLM) is a fast inference engine for large language models, designed for high throughput and low latency. It clusters, etc. It's a com

### Streamed generation

In [6]:
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    stream=True,
)

section = None

for chunk in stream:
    delta = chunk.choices[0].delta
    if not delta:
        continue

    reasoning = getattr(delta, "reasoning_content", None)
    if reasoning:
        if section != "reasoning":
            print("Reasoning: ", end="", flush=True)
            section = "reasoning"
        print(reasoning, end="", flush=True)

    if delta.content:
        if section != "content":
            print("\n\nContent: ", end="", flush=True)
            section = "content"
        print(delta.content, end="", flush=True)

Reasoning: Here, the user is asking for the first 5 prime numbers. A simple question about prime numbers. The first 5 prime numbers are 2, 3, 5, 7, 11. But let me double-check. Prime numbers are numbers greater than 1 that have exactly two factors: 1 and themselves. Starting from 2: 2 (prime), 3 (prime), 4 (not prime, divisible by 2), 4 is not prime, 5 is prime, 7 is prime, 11 is prime. So the first 5 primes are 2, 3, 5, 7, 11. I should provide these in a clear format. Maybe list them out. The user wants the first 5 prime numbers, so I'll list them. I'll output just the answer, maybe with a brief explanation. The instructions say: "You dont need to write a concise and direct title." The user is asking "What are the first 5 prime numbers?" So I need to output the numbers. I'll just list them: 2, 3, 5, 7, 11. I'll format as requested.

Content: The first 5 prime numbers are 2, 3, 5, 7, and 11.

### Reasoning

> **Note:** The model supports two modes — Reasoning ON (default) and Reasoning OFF. Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`, as shown below.

In [8]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about SGLang."}
    ],
    temperature=0.2,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False, "force_nonempty_content": True}}
)
print("Content:", resp.choices[0].message.content)

Reasoning on
Reasoning: Here are questions from the Lyft users:  
1. Write a haiku about GPUs.
2. Check the content for writing. Possibly about proofreading.
3. Provide a solution that can include verified answers.
4. I need to get to the image generator

Respond with the exact answer for writing a haiku about GPUs only.
Content: Coils of silicon glow,
pipelines feed data streams deep,
reality-rig rendered.

Reasoning off
Content: Here are 3 interesting facts about SGLang:

**1. It was developed at the University of Washington.**
SGLang was created by researchers from the Universitying of Washington and the S-Lab at NVIDIA. The project was born out of the same group that created the original DeepSpeed and Megatron-LM projects.

**2. It introduces "wild" batch size accumulation.**
One of the standout features of SGLang is its approach to memory management. It uses a technique that allows it to handle "wildly different sequence lengths and batch sizes efficiently, making it particularly 

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`.

> **Note:** When using tool calling with reasoning enabled, you must pass `"force_nonempty_content": true` inside `chat_template_kwargs`. Without it, the server may not correctly parse both the reasoning trace and the tool call output together.

In [2]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    stream=False,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": True,
            "force_nonempty_content": True,
        }
    }
)

choice = completion.choices[0]
print("Reasoning:", choice.message.reasoning_content)
print("Content:", choice.message.content)
print("Tool calls:", choice.message.tool_calls)

Reasoning: Here's a thinking process:

1.  **Analyze User Input:**
   - Bill total: $50
   - Tip percentage: 15%
   - Question: What will be the amount for 15% tip?

2.  **Identify Required Tool:**
   - The `calculate_tip` function takes `bill_total` and `tip_percentage` as parameters.
   - Both are provided: `bill_total` = 50, `tip_percentage` = 15.

3.  **Check Parameter Requirements:**
   - `bill_type`: integer (50)
   - `tip_percentage_type`: integer (15)
   - Both are required and match the input.

4.  **Execute Tool Call:**
   - Call `calculate_tip` with `bill_total=50` and `tip_percentage=15`.

5.  **Formulate Response:**
   - Wait for the tool result, then present it to the user.
   - Actually, I can just call the tool now.

Let's do it.⟩

Content: 

Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_c4d4ee7028d84052858dee80', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function', index=0)]


### Controlling Reasoning Budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [18]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning_content or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [19]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"  # use actual HF model ID for tokenizer
)

In [20]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    max_tokens=1024,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: Here's a haiku about GPUs:

Through silicon's graceful dance,
intense math in blocks of light,
triple-slash, through the night..
Content: Broadbending wires blaze,  
halo cooling fans spin fast, GeForce glows.


## Cleanup and shutdown

To free resources after this notebook:

1. Stop the SGLang server in the terminal where it was started (`Ctrl+C`).
2. Optionally run the next cell to clear notebook-side CUDA cache.
3. Restart the kernel if needed to ensure a clean state.

In [21]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")
print("Primary teardown step: stop the SGLang server with Ctrl+C in its terminal.")

Notebook-side CUDA cache cleanup complete.
Primary teardown step: stop the SGLang server with Ctrl+C in its terminal.
